# SL delay data: exploration and pipeline checks

Reads the local `data/` folder (bronze, silver and gold written by the scripts and Spark jobs) with pandas/pyarrow, so it runs on a laptop or in the `jupyter` container.

1. What a real-time snapshot contains
2. Why a stop visit must be deduplicated, shown on one real trip
3. Pipeline row counts (QA)

In [1]:
from pathlib import Path
import pandas as pd
import pyarrow.dataset as ds

ROOT = Path.cwd() if (Path.cwd() / "data").exists() else Path.cwd().parent
DATA = ROOT / "data"
DAY = "2026-09-15"
pd.set_option("display.width", 160)

## 1. Bronze: one row per stop update per snapshot

`scripts/rt_to_parquet.py` flattens every GTFS-RT TripUpdates snapshot (one every ~20 s) into one row per `stop_time_update`. Times are unix epoch UTC; `start_date` is the service date.

In [2]:
bronze = ds.dataset(DATA / "bronze" / "tripupdates" / f"file_day={DAY}")
print(f"{bronze.count_rows():,} stop updates on {DAY}")
bronze.head(5).to_pandas()

61,261,614 stop updates on 2026-09-15


,snapshot_ts,trip_id,start_date,trip_sched_rel,vehicle_id,stop_id,stop_sequence,stop_sched_rel,arr_delay,arr_time,arr_uncertainty,dep_delay,dep_time
0,1789423192,14010000729005087,20260914,0,9031001004505120,9022001063969001,61,0,-72,1789422551,0,-72,1789422551
1,1789423192,14010000729005087,20260914,0,9031001004505120,9022001063999001,62,0,-99,1789422600,0,-99,1789422600
2,1789423192,14010000729005087,20260914,0,9031001004505120,9022001063971001,63,0,-125,1789422642,0,-125,1789422642
3,1789423192,14010000729005087,20260914,0,9031001004505120,9022001063973001,64,0,-125,1789422674,0,-125,1789422674
4,1789423192,14010000729005087,20260914,0,9031001004505120,9022001064173002,65,0,-121,1789422735,0,-121,1789422735


## 2. Deduplication: from predictions to one observed arrival

Each snapshot repeats the latest prediction for all upcoming stops of a trip, and keeps a stop for ~10 minutes after the vehicle has passed it. The reports of one stop visit, over time, converge to the observed arrival. The silver job keeps the **last** report of each (`trip_id`, `start_date`, `stop_sequence`) and accepts it as observed when its arrival time is not later than that last snapshot.

In [3]:
silver = pd.read_parquet(DATA / "silver" / "stop_events" / f"service_date={DAY}")
example = silver[(silver["mode"] == "Metro") & (silver["n_reports"] > 50)].iloc[0]
example[["trip_id", "line", "line_name", "station_name", "stop_sequence", "delay_s", "n_reports"]]

trip_id          14010000702426922
line                            19
line_name             Gröna linjen
station_name                 Alvik
stop_sequence                   23
delay_s                         60
n_reports                      171
Name: 709, dtype: object

In [4]:
f = (ds.field("trip_id") == example.trip_id) & (ds.field("stop_sequence") == int(example.stop_sequence))
reports = bronze.to_table(filter=f).to_pandas().sort_values("snapshot_ts")
reports["snapshot_utc"] = pd.to_datetime(reports["snapshot_ts"], unit="s")
reports["predicted_arrival_utc"] = pd.to_datetime(reports["arr_time"], unit="s")
reports["vehicle_passed"] = reports["arr_time"] <= reports["snapshot_ts"]
print(f"{len(reports)} reports for this one stop visit; the last one is the observed delay ({example.delay_s} s)")
reports[["snapshot_utc", "predicted_arrival_utc", "arr_delay", "vehicle_passed"]].iloc[::max(1, len(reports) // 15)]

171 reports for this one stop visit; the last one is the observed delay (60 s)


,snapshot_utc,predicted_arrival_utc,arr_delay,vehicle_passed
0,2026-09-15 05:27:51,2026-09-15 06:12:48,-30,False
11,2026-09-15 05:31:30,2026-09-15 06:12:48,-30,False
22,2026-09-15 05:35:04,2026-09-15 06:12:48,-30,False
33,2026-09-15 05:38:43,2026-09-15 06:13:06,-12,False
44,2026-09-15 05:42:37,2026-09-15 06:12:57,-21,False
55,2026-09-15 05:46:17,2026-09-15 06:12:57,-21,False
66,2026-09-15 05:49:56,2026-09-15 06:12:57,-21,False
77,2026-09-15 05:53:33,2026-09-15 06:12:57,-21,False
88,2026-09-15 05:57:06,2026-09-15 06:12:57,-21,False
99,2026-09-15 06:00:35,2026-09-15 06:13:12,-6,False


In [5]:
import plotly.express as px

fig = px.line(reports, x="snapshot_utc", y="arr_delay", line_shape="hv", height=320,
              labels={"snapshot_utc": "snapshot time (UTC)", "arr_delay": "predicted delay (s)"},
              title=f"Predicted delay of one stop visit over time ({example.line_name} {example.line}, {example.station_name})")
fig.add_vline(x=reports.loc[reports["vehicle_passed"], "snapshot_utc"].min(), line_dash="dash", line_color="gray",
              annotation_text="vehicle passed")
fig.update_layout(template="plotly_white")
fig

## 3. Pipeline row counts

From `silver/qa` (copied to `gold/qa`). Averaging bronze rows directly would weight each arrival by how long it stayed in the feed, and mix predictions with observations.

In [6]:
qa = pd.read_parquet(DATA / "gold" / "qa")
qa

,step,n_rows
0,bronze stop updates,777382896
1,stop visits (all service dates),8312905
2,stop visits in period,8270405
3,not observed (left the feed early),74550
4,added/canceled trip or skipped stop,19359
5,no delay value,0
6,|delay| > 2 h,507
7,observed events,8175989
8,trip_id not in static GTFS,47
9,stop_id not in static GTFS,0


In [7]:
by_day_type = pd.read_parquet(DATA / "gold" / "by_day_type")
by_day_type[by_day_type["day_type"] != "All"].pivot(index="mode", columns="day_type", values="pct_delayed")

day_type,weekday,weekend
mode,,
All,19.63,23.04
Bus,21.75,26.58
Commuter rail,7.89,3.34
Ferry,22.53,19.73
Metro,2.27,0.82
Tram,3.37,2.87
